# Pipeline auditing & live parameter tuning

This notebook shows how to **see what each stage of a patching pipeline throws out**, and then **tune the filters with sliders** until you like the result.

The workflow:
1. Build a pipeline as usual.
2. `p.dry_run()` — runs the filters **without writing any output** (no encoding, no files). Fast to iterate.
3. `p.print_audit()` — a per-stage keep/drop funnel table.
4. `visualize_audit(p, slide)` — an **interactive overlay**: kept patches in green, dropped patches colored by the stage that removed them, plus a **slider per tunable parameter**.

Dragging a slider re-decides keep/drop for every patch *in the browser* — no Python re-run — and prints a config you can paste straight back into your pipeline.

In [ ]:
from wsi_patching import OtsuFilter, PatchExtractor, RemoveEdgeTiles, WSIGrid, visualize_audit, LowContrastBackgroundFilter

# A small slide shipped in the repo. Swap for your own path.
SLIDE = "../data/RBIO-GC072-HE-01.tiff"

## 1. Build a pipeline and dry-run it

No writer is needed for a dry run. `resolution=2` patches at a downsampled level so the demo is quick.

In [ ]:
p = (
    WSIGrid(slides=[SLIDE], resolution=2, unit="level", use_gpu=False)
    .then(PatchExtractor(tile_size=224, stride=224))
    .then(RemoveEdgeTiles(depth=1))               # drop the border ring of tiles
    .then(LowContrastBackgroundFilter())  # drop patches with too little contrast
    .then(OtsuFilter(min_tissue_fraction=0.35))   # drop patches with too little tissue
)

p.dry_run(num_workers=1)
p.print_audit()

## 2. The interactive overlay

Green = kept. Each other color = a stage that dropped those patches. **Hover** a patch to see its stage + metadata (e.g. `tissue_fraction`).

Three ways to tune, all live and none of them re-running Python:

- **Drag a slider**, or **type an exact value** in the box on the right (`min_tissue_fraction`, `range_threshold`, `depth`).
- **Uncheck a stage** to see the pipeline *without* it — the patches it had dropped are re-judged by the remaining stages and recolored.
- **Click ▲▼** to reorder the filters.

The overlay, the funnel table and the config box all update together. When you're happy, copy the config box straight into the pipeline above — it's emitted in the order you chose.

> **Reordering never changes which patches survive** (a patch is kept only if it passes every filter). It changes *who gets the blame* and *how much work each stage does*. Move `OtsuFilter` to the front and watch: it still keeps the same 10 patches, but you'll see it was really rejecting 167 — and that running it last means it only has to process 67 patches instead of 252.

In [ ]:
report = visualize_audit(p, SLIDE)
report  # renders inline